<a href="https://colab.research.google.com/github/caramos84/QC_Video/blob/main/notebooks/04_DecisionEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QC Video - Notebook 04
## QC Decision Engine

Este notebook ya NO consolida datos a mano: es un wrapper delgado sobre el
paquete Python `rules/` (el Motor de QC real). Toda la lógica de reglas,
scoring y generación de reporte vive en `rules/*.py` y está cubierta por
`pytest` — ver `tests/`.

Flujo:

1. Clonar el repo (para tener acceso a `rules/`, `profiles/`, `samples/`).
2. Cargar `asset_knowledge.json` (el que genera el pipeline de Notebooks 01-03).
3. Elegir un `profile` (canal/placement, ej. `meta_instagram_reels`) y,
   opcionalmente, un `brief` de campaña.
4. Ejecutar `rules.engine.run_qc(...)`.
5. Generar y descargar `qc_report.json`, `qc_score.json`, `qc_summary.md`.

Entradas:

- `asset_knowledge.json` (o un `.zip` que lo contenga)
- `profile_id` (ver `profiles/`)
- opcional: un `brief.yaml` de campaña (ver `samples/briefs/`)

Salidas:

- `outputs/<asset_id>/qc_report.json`
- `outputs/<asset_id>/qc_score.json`
- `outputs/<asset_id>/qc_summary.md`

## 1. Clonar el repo e instalar el paquete `rules/`

In [ ]:
import os

REPO_URL = "https://github.com/caramos84/QC_Video.git"
REPO_DIR = "/content/QC_Video"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

## 2. Cargar `asset_knowledge.json`

Acepta el archivo `asset_knowledge.json` directo, o un `.zip` que lo contenga
(por ejemplo el `asset_knowledge.zip` que descargaba la versión anterior de
este notebook).

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
uploaded_name = next(iter(uploaded.keys()))

WORKDIR = Path("/content/qc_run")
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True)

if uploaded_name.lower().endswith(".zip"):
    with zipfile.ZipFile(uploaded_name, "r") as z:
        z.extractall(WORKDIR)
    candidates = list(WORKDIR.rglob("asset_knowledge.json"))
    if not candidates:
        raise FileNotFoundError("El zip no contiene asset_knowledge.json")
    ASSET_KNOWLEDGE_PATH = candidates[0]
else:
    ASSET_KNOWLEDGE_PATH = WORKDIR / "asset_knowledge.json"
    shutil.move(uploaded_name, ASSET_KNOWLEDGE_PATH)

print(f"asset_knowledge.json listo en: {ASSET_KNOWLEDGE_PATH}")

## 3. Elegir profile (canal/placement) y brief opcional

In [ ]:
from pathlib import Path

PROFILES_DIR = Path(REPO_DIR) / "profiles"
print("Profiles disponibles:")
for p in sorted(PROFILES_DIR.glob("*.yaml")):
    print(" -", p.stem)

In [ ]:
# Editar antes de correr:
PROFILE_ID = "meta_instagram_reels"
BRIEF_PATH = Path(REPO_DIR) / "samples" / "briefs" / "example_campaign_brief.yaml"  # o None

PROFILE_PATH = PROFILES_DIR / f"{PROFILE_ID}.yaml"
assert PROFILE_PATH.exists(), f"No existe el profile: {PROFILE_PATH}"
print("Profile:", PROFILE_PATH)
print("Brief:", BRIEF_PATH if BRIEF_PATH else "(ninguno)")

## 3.5 Análisis semántico (LLM)

Compara el texto OCR (`visual.frames[].text`) y la transcripción de audio
(`audio.transcript.text`) contra las dimensiones semánticas configuradas en
el brief (`semantic.key_messages`, `semantic.brand_positioning_statement`,
`semantic.guidelines_text`, `legal.disclaimer_required`), vía Gemini 3.6
Flash (`google-genai`, API key en el secret de Colab `GEMINI_API_KEY`).

Solo se llama al LLM para las dimensiones que el brief configura (ahorra
costo); si el brief no configura ninguna, se omite la llamada por completo.
Cualquier error (secret ausente, red, respuesta malformada) se captura y
deja `semantic` ausente del `asset_knowledge.json` -- las 3 reglas
`SEMANTICA_*` quedan `NOT_EVALUATED`, nunca crashea el notebook.

In [ ]:
!pip install -q -U "google-genai>=2.3.0"

In [ ]:
import json
import yaml
from pathlib import Path

asset_knowledge = json.loads(Path(ASSET_KNOWLEDGE_PATH).read_text(encoding="utf-8"))

brief_doc = None
if BRIEF_PATH:
    brief_doc = yaml.safe_load(Path(BRIEF_PATH).read_text(encoding="utf-8"))

semantic_cfg = (brief_doc or {}).get("semantic") or {}
legal_cfg = (brief_doc or {}).get("legal") or {}

key_messages = semantic_cfg.get("key_messages") or []
brand_positioning_statement = semantic_cfg.get("brand_positioning_statement")
guidelines_text = semantic_cfg.get("guidelines_text")
disclaimer_required = legal_cfg.get("disclaimer_required")
disclaimer_text_contains = legal_cfg.get("disclaimer_text_contains") or []

want_message_compliance = bool(key_messages)
want_brand_positioning = bool(brand_positioning_statement)
want_guideline_compliance = bool(guidelines_text) or bool(disclaimer_required)

frames = ((asset_knowledge.get("visual") or {}).get("frames")) or []
ocr_lines = [t for f in frames for t in (f.get("text") or [])]
transcript_text = (((asset_knowledge.get("audio") or {}).get("transcript")) or {}).get("text") or ""

semantic_prompt = None
if not (want_message_compliance or want_brand_positioning or want_guideline_compliance):
    print("Brief no configura ninguna dimensión semántica -- se omite la llamada a Gemini.")
else:
    prompt_parts = [
        "Eres un revisor de cumplimiento de marca para creatividades publicitarias de video.",
        "Texto detectado por OCR en los frames del video:",
        "\n".join(f"- {t}" for t in ocr_lines) or "(sin texto OCR detectado)",
        "",
        "Transcripción del audio:",
        transcript_text or "(sin transcripción)",
        "",
    ]
    if want_message_compliance:
        prompt_parts += [
            "## Evaluación 1: cumplimiento de mensaje",
            "Mensajes clave que el brief exige comunicar:",
            "\n".join(f"- {m}" for m in key_messages),
            "Completa 'message_compliance': compliant=true si el OCR+transcripción comunican "
            "razonablemente estos mensajes (no requiere texto literal exacto); matched_messages "
            "con los detectados, missing_messages con los que no.",
            "",
        ]
    if want_brand_positioning:
        prompt_parts += [
            "## Evaluación 2: posicionamiento de marca",
            f'Statement de posicionamiento de marca del brief: "{brand_positioning_statement}"',
            "Completa 'brand_positioning': compliant=true si el tono/mensaje comunicado es "
            "consistente con ese posicionamiento.",
            "",
        ]
    if want_guideline_compliance:
        prompt_parts.append("## Evaluación 3: cumplimiento de lineamientos")
        if guidelines_text:
            prompt_parts.append(f'Lineamientos generales del brief: "{guidelines_text}"')
        if disclaimer_required:
            prompt_parts.append(
                "Este brief EXIGE un disclaimer legal. Frases aproximadas que debería contener "
                f"(no exactas): {disclaimer_text_contains or '(no especificadas; solo verificar que exista algún disclaimer)'}"
            )
        prompt_parts.append(
            "Completa 'guideline_compliance': compliant=true si se cumplen los lineamientos Y "
            "(si aplica) el disclaimer. Completa 'disclaimer_check' SOLO si se pidió evaluar un "
            "disclaimer arriba (required según si el brief lo exige, found=true si el "
            "OCR/transcript contiene un disclaimer razonablemente equivalente, matched_text con "
            "el texto encontrado o null)."
        )
    prompt_parts.append(
        "\nIMPORTANTE: completa SOLO los campos de las evaluaciones solicitadas arriba; deja los "
        "demás campos del JSON en null."
    )
    semantic_prompt = "\n".join(prompt_parts)
    print(semantic_prompt)

In [ ]:
from typing import Optional
from pydantic import BaseModel


class MessageCompliance(BaseModel):
    compliant: bool
    reasoning: str
    matched_messages: list[str]
    missing_messages: list[str]


class BrandPositioning(BaseModel):
    compliant: bool
    reasoning: str


class DisclaimerCheck(BaseModel):
    required: bool
    found: bool
    matched_text: Optional[str] = None


class GuidelineCompliance(BaseModel):
    compliant: bool
    reasoning: str
    disclaimer_check: Optional[DisclaimerCheck] = None


class SemanticAnalysis(BaseModel):
    message_compliance: Optional[MessageCompliance] = None
    brand_positioning: Optional[BrandPositioning] = None
    guideline_compliance: Optional[GuidelineCompliance] = None


if semantic_prompt is not None:
    try:
        from google import genai
        from google.colab import userdata

        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
        if not GEMINI_API_KEY:
            raise RuntimeError("Colab secret 'GEMINI_API_KEY' no configurado")

        client = genai.Client(api_key=GEMINI_API_KEY)
        interaction = client.interactions.create(
            model="gemini-3.6-flash",
            input=semantic_prompt,
            response_format=[
                {
                    "type": "text",
                    "mime_type": "application/json",
                    "schema": SemanticAnalysis.model_json_schema(),
                }
            ],
        )
        parsed = SemanticAnalysis.model_validate_json(interaction.output_text)

        semantic_result = {}
        if want_message_compliance and parsed.message_compliance is not None:
            semantic_result["message_compliance"] = parsed.message_compliance.model_dump()
        if want_brand_positioning and parsed.brand_positioning is not None:
            semantic_result["brand_positioning"] = parsed.brand_positioning.model_dump()
        if want_guideline_compliance and parsed.guideline_compliance is not None:
            semantic_result["guideline_compliance"] = parsed.guideline_compliance.model_dump()

        if semantic_result:
            asset_knowledge["semantic"] = semantic_result
            Path(ASSET_KNOWLEDGE_PATH).write_text(
                json.dumps(asset_knowledge, ensure_ascii=False, indent=2), encoding="utf-8"
            )
            print(f"Análisis semántico guardado en {ASSET_KNOWLEDGE_PATH}: {list(semantic_result.keys())}")
        else:
            print("Gemini no devolvió ninguna de las dimensiones solicitadas; 'semantic' queda ausente.")
    except Exception as e:
        print(f"⚠️ Análisis semántico omitido por error: {e}")

## 4. Ejecutar el Motor de QC

In [ ]:
from rules.engine import run_qc_from_paths

report = run_qc_from_paths(
    ASSET_KNOWLEDGE_PATH,
    PROFILE_PATH,
    BRIEF_PATH,
)

print(f"Veredicto: {report.score_summary.verdict.value}")
print(f"Score: {report.score_summary.score}/100")
print(f"Cobertura: {report.score_summary.coverage_pct}%")

## 5. Generar artefactos de salida

`qc_report.json` (findings detallados), `qc_score.json` (score/veredicto) y
`qc_summary.md` (resumen legible, con la sección de cobertura y brechas).

In [ ]:
from rules.report import write_report

asset_id_safe = report.asset_id.replace("/", "_")
output_dir = Path("/content/qc_run/outputs") / asset_id_safe
paths = write_report(report, output_dir)

for name, path in paths.items():
    print(f"{name}: {path}")

## 6. Mostrar el resumen

In [ ]:
from IPython.display import Markdown, display

display(Markdown(paths["qc_summary"].read_text(encoding="utf-8")))

## 7. Descargar los artefactos generados

In [ ]:
import shutil as _shutil
from google.colab import files as _files

zip_base = f"/content/qc_output_{asset_id_safe}"
_shutil.make_archive(zip_base, "zip", output_dir)
_files.download(f"{zip_base}.zip")

## Resultado final

Este notebook ya no genera `asset_knowledge.json` (eso lo sigue haciendo la
consolidación previa) — genera el veredicto real de QC:

```text
outputs/<asset_id>/
├── qc_report.json
├── qc_score.json
└── qc_summary.md
```

Toda la lógica (reglas, severidades, scoring, NOT_EVALUATED por datos
faltantes) vive en `rules/*.py` y está testeada con `pytest tests/` — este
notebook es solo la interfaz interactiva sobre esa lógica.